# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

## 1. Set Up Paths and the Vector Store

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/sujat/projects/AI-Main/Advanced_Rag_Codes/04_vector_stores')

In [3]:
# Load environment variables from the local .env file.
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

ValueError: Please add your OPENAI_API_KEY to the .env file before running this notebook.

In [4]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\04_vector_stores\notebooks\chroma_langchain_db


In [5]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [6]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),#without this vector store will be created inside RAM
)

print("Vector store is ready.")

c:\Users\sujat\projects\AI-Main\Advanced_Rag_Codes\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 789.79it/s]


Vector store is ready.


## 2. Add Small Helper Functions

In [7]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [8]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [9]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [16]:
print(uuid4())

3296d718-9443-40df-b339-f8a7e8a5719e


In [17]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=7cf7bcc8-dea2-43b6-a608-780d66293a2d
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=46bed25f-953e-4738-bc34-0cbb9d7fb38c
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=cbce6cdd-f93c-44d2-9c1c-b0d46f869418
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=d4097243-4401-4d6e-bea9-c70d0d183ecd
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=8c644094-9909-4ffe-bf89-e5c8e552d946
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=3767ea87-210f-4a4a-9420-fd58a22ec675
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [18]:
documents[0].id

'7cf7bcc8-dea2-43b6-a608-780d66293a2d'

In [19]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
7cf7bcc8-dea2-43b6-a608-780d66293a2d
46bed25f-953e-4738-bc34-0cbb9d7fb38c
cbce6cdd-f93c-44d2-9c1c-b0d46f869418
d4097243-4401-4d6e-bea9-c70d0d183ecd
8c644094-9909-4ffe-bf89-e5c8e552d946
3767ea87-210f-4a4a-9420-fd58a22ec675
7b333fec-466c-4417-8edf-bb565db5673a
fb311072-e248-4602-9914-16e54ed6f199
cf2f832f-7924-4712-8a5c-56110ad68f28
29e7ab8f-8d34-4020-956a-61584e79f156

Total inserted documents: 10


## 4. Read the Stored Data Back

In [20]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [21]:
raw_records

{'ids': ['7cf7bcc8-dea2-43b6-a608-780d66293a2d',
  '46bed25f-953e-4738-bc34-0cbb9d7fb38c',
  'cbce6cdd-f93c-44d2-9c1c-b0d46f869418',
  'd4097243-4401-4d6e-bea9-c70d0d183ecd',
  '8c644094-9909-4ffe-bf89-e5c8e552d946',
  '3767ea87-210f-4a4a-9420-fd58a22ec675',
  '7b333fec-466c-4417-8edf-bb565db5673a',
  'fb311072-e248-4602-9914-16e54ed6f199',
  'cf2f832f-7924-4712-8a5c-56110ad68f28',
  '29e7ab8f-8d34-4020-956a-61584e79f156'],
 'embeddings': array([[ 0.00980197, -0.0154417 ,  0.05125516, ...,  0.13042368,
          0.05186963, -0.08871341],
        [-0.06569786,  0.01161643, -0.01263002, ...,  0.07050049,
          0.06507605, -0.04185101],
        [-0.02644424, -0.01801335,  0.00899873, ...,  0.09935074,
          0.12936939, -0.05477072],
        ...,
        [ 0.03539194,  0.06386086,  0.02307184, ...,  0.13626683,
         -0.01406124, -0.01717238],
        [ 0.02758242,  0.04555592, -0.05576294, ..., -0.02275212,
          0.05048471, -0.05864125],
        [-0.02807554,  0.03799979, 

In [22]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [23]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
7cf7bcc8-dea2-43b6-a608-780d66293a2d
46bed25f-953e-4738-bc34-0cbb9d7fb38c
cbce6cdd-f93c-44d2-9c1c-b0d46f869418


In [24]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['fb311072-e248-4602-9914-16e54ed6f199',
 'cf2f832f-7924-4712-8a5c-56110ad68f28',
 '29e7ab8f-8d34-4020-956a-61584e79f156']

In [25]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=fb311072-e248-4602-9914-16e54ed6f199
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=cf2f832f-7924-4712-8a5c-56110ad68f28
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=29e7ab8f-8d34-4020-956a-61584e79f156
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [26]:
print(selected_documents)

[Document(id='fb311072-e248-4602-9914-16e54ed6f199', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='cf2f832f-7924-4712-8a5c-56110ad68f28', metadata={'topic': 'Cricket', 'doc_number': 9}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='29e7ab8f-8d34-4020-956a-61584e79f156', metadata={'topic': 'Cricket', 'doc_number': 10}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [27]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [28]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=d4097243-4401-4d6e-bea9-c70d0d183ecd
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=fb311072-e248-4602-9914-16e54ed6f199
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=7b333fec-466c-4417-8edf-bb565db5673a
   topic=LLM | doc_number=7
   content=LLMs generate text by predicting likely next tokens from patterns learned during training.



In [29]:
search_results

[Document(id='d4097243-4401-4d6e-bea9-c70d0d183ecd', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='fb311072-e248-4602-9914-16e54ed6f199', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='7b333fec-466c-4417-8edf-bb565db5673a', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.')]

In [30]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='d4097243-4401-4d6e-bea9-c70d0d183ecd', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.7122337818145752),
 (Document(id='fb311072-e248-4602-9914-16e54ed6f199', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  1.0460386276245117),
 (Document(id='7b333fec-466c-4417-8edf-bb565db5673a', metadata={'doc_number': 7, 'topic': 'LLM'}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.2469239234924316),
 (Document(id='3767ea87-210f-4a4a-9420-fd58a22ec675', metadata={'topic': 'RAG', 'doc_number': 6}, page_content='Vector stores are important in RAG because they make semantic search over embedded documents possible.'),
  1.2675285339355469)]

score is distance here, less is better

## 6. Update Existing Documents

In [31]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['d4097243-4401-4d6e-bea9-c70d0d183ecd',
 'fb311072-e248-4602-9914-16e54ed6f199']

In [33]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=d4097243-4401-4d6e-bea9-c70d0d183ecd
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=fb311072-e248-4602-9914-16e54ed6f199
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [34]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [35]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
d4097243-4401-4d6e-bea9-c70d0d183ecd
fb311072-e248-4602-9914-16e54ed6f199


In [36]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=d4097243-4401-4d6e-bea9-c70d0d183ecd
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=fb311072-e248-4602-9914-16e54ed6f199
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [37]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [38]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=d4097243-4401-4d6e-bea9-c70d0d183ecd
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=8c644094-9909-4ffe-bf89-e5c8e552d946
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [39]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['cf2f832f-7924-4712-8a5c-56110ad68f28',
 '29e7ab8f-8d34-4020-956a-61584e79f156']

In [40]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
cf2f832f-7924-4712-8a5c-56110ad68f28
29e7ab8f-8d34-4020-956a-61584e79f156


In [41]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
7cf7bcc8-dea2-43b6-a608-780d66293a2d
46bed25f-953e-4738-bc34-0cbb9d7fb38c
cbce6cdd-f93c-44d2-9c1c-b0d46f869418
d4097243-4401-4d6e-bea9-c70d0d183ecd
8c644094-9909-4ffe-bf89-e5c8e552d946
3767ea87-210f-4a4a-9420-fd58a22ec675
7b333fec-466c-4417-8edf-bb565db5673a
fb311072-e248-4602-9914-16e54ed6f199

Deleted ids still present?
cf2f832f-7924-4712-8a5c-56110ad68f28: False
29e7ab8f-8d34-4020-956a-61584e79f156: False


In [42]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
